In [1]:
!pip install opencv-python numpy pandas matplotlib pillow tqdm -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\sujal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import os
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm


JSON_PATH = r"C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset\kaggle\train\train.json"

IMAGE_DIR = r"C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset\kaggle\train\images"

PROJECT_DIR = r"C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset"

MASK_DIR = os.path.join(
    PROJECT_DIR,
    "masks"
)

SPINE_DIR = os.path.join(
    PROJECT_DIR,
    "spines"
)

VIS_DIR = os.path.join(
    PROJECT_DIR,
    "visualizations"
)

METADATA_PATH = os.path.join(
    PROJECT_DIR,
    "metadata.csv"
)

os.makedirs(
    MASK_DIR,
    exist_ok=True
)

os.makedirs(
    SPINE_DIR,
    exist_ok=True
)

os.makedirs(
    VIS_DIR,
    exist_ok=True
)

print("JSON:", JSON_PATH)
print("Images:", IMAGE_DIR)
print("Masks:", MASK_DIR)
print("Spines:", SPINE_DIR)
print("Visualizations:", VIS_DIR)

JSON: C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset\kaggle\train\train.json
Images: C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset\kaggle\train\images
Masks: C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset\masks
Spines: C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset\spines
Visualizations: C:\Users\sujal\OneDrive\Documents\praccodes\solar-filament-project\dataset\visualizations


In [8]:
with open(
    JSON_PATH,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


print("Top-level keys:")
print(data.keys())

print()

print(
    "Number of image records:",
    len(data.get("images", []))
)

print(
    "Number of annotations:",
    len(data.get("annotations", []))
)

print()

if "categories" in data:

    print("Categories:")

    for category in data["categories"]:

        print(
            category
        )

Top-level keys:
dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])

Number of image records: 1154
Number of annotations: 8199

Categories:
{'supercategory': 'filament', 'id': 1, 'name': 'Left'}
{'supercategory': 'filament', 'id': 2, 'name': 'Right'}
{'supercategory': 'filament', 'id': 3, 'name': 'Unidentifiable'}
{'supercategory': 'filament', 'id': 4, 'name': 'Ambiguous'}


In [9]:
annotation = data["annotations"][0]

print(
    "Annotation keys:"
)

print(
    annotation.keys()
)

print()

print(
    "Image ID:",
    annotation.get("image_id")
)

print(
    "Area:",
    annotation.get("area")
)

print(
    "Bounding box:",
    annotation.get("bbox")
)

print(
    "Category:",
    annotation.get("category_id")
)

print(
    "Has segmentation:",
    "segmentation" in annotation
)

print(
    "Has spine:",
    "spine" in annotation
)

print()

if "segmentation" in annotation:

    print(
        "Segmentation type:",
        type(
            annotation["segmentation"]
        )
    )

if "spine" in annotation:

    print(
        "Spine length:",
        len(
            annotation["spine"]
        )
    )

Annotation keys:
dict_keys(['segmentation', 'area', 'iscrowd', 'spine', 'image_id', 'bbox', 'category_id', 'id'])

Image ID: 040301-20140609195854Bh
Area: 750.0
Bounding box: [388.4121, 799.966, 92.52639999999997, 52.485000000000014]
Category: 2
Has segmentation: True
Has spine: True

Segmentation type: <class 'list'>
Spine length: 54


In [11]:
images = data["images"]

image_lookup = {}

for image in images:

    image_id = image["id"]

    image_lookup[
        image_id
    ] = image


print(
    "Images indexed:",
    len(image_lookup)
)

print()

first_id = list(
    image_lookup.keys()
)[0]

print(
    "Example image ID:",
    first_id
)

print(
    "Image information:"
)

print(
    image_lookup[first_id]
)

Images indexed: 1154

Example image ID: 040301-20140609195854Bh
Image information:
{'license': 1, 'file_name': '20140609195854Bh.jpeg', 'url': 'https://gong2.nso.edu/HA/hag/201406/20140609/20140609195854Bh.jpg', 'height': 2048, 'width': 2048, 'date_captured': '2014-06-09 19:58:54', 'id': '040301-20140609195854Bh'}


In [12]:
from collections import defaultdict


annotations_by_image = defaultdict(
    list
)


for annotation in data["annotations"]:

    image_id = annotation[
        "image_id"
    ]

    annotations_by_image[
        image_id
    ].append(
        annotation
    )


print(
    "Images with annotations:",
    len(
        annotations_by_image
    )
)


for image_id in list(
    annotations_by_image.keys()
)[:5]:

    print(
        image_id,
        "→",
        len(
            annotations_by_image[
                image_id
            ]
        ),
        "filaments"
    )

Images with annotations: 1154
040301-20140609195854Bh → 12 filaments
050103-20111116063134Lh → 11 filaments
050102-20111116063134Lh → 11 filaments
050101-20111116063134Lh → 11 filaments
030101-20130725122934Ch → 16 filaments
